# 01 — Data Audit

## 1. Environment & Paths Setup

This section handles:
- **Environment Detection**: Automatically identifies whether the notebook is running on **Google Colab** or a **Local** machine.
  - On Colab: Mounts Google Drive for persistent storage.
  - On Local: Resolves the project root path dynamically whether executed from the root directory or inside `notebooks/`.
- **Directory Initialization**: Configures key directory paths (`data/raw` and `data/audit`) and creates them automatically if they do not exist.

In [14]:
import importlib.util

try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except (ModuleNotFoundError, ImportError):
    IS_COLAB = False

if IS_COLAB:
    # Google Colab: use Google Drive for persistent storage
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/Home/10_Projects/plant-disease-classification"
    )
else:
    # Local: locate the project root from the notebook
    PROJECT_ROOT = Path.cwd()

    # Support running from either the project root or notebooks/
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
AUDIT_DIR = DATA_DIR / "audit"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Colab' if IS_COLAB else 'Local'}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset directory: {RAW_DATA_DIR}")


Environment: Local
Project root: /home/tanthanh/Documents/Projects/PlantDiseaseClassification
Dataset directory: /home/tanthanh/Documents/Projects/PlantDiseaseClassification/data/raw


## 2. Dataset Acquisition

Download and extract the dataset into `RAW_DATA_DIR`.

- **Source**: Kaggle API (or custom URL).
- **Behavior**: Checks if data already exists locally to avoid redundant downloads, unpacks archives automatically, and cleans up temporary zip files.

In [15]:
!pip install kaggle

In [16]:
if IS_COLAB:
    import os
    from google.colab import userdata

    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

    print("Kaggle credentials configured.")
else:
    # Local: ensure Kaggle API credentials are set up
    print("Ensure Kaggle API credentials are set up")
    pass

Ensure Kaggle API credentials are set up


In [17]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p {RAW_DATA_DIR} --unzip --force

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
 19%|███████                               | 390M/2.04G [00:21<01:32, 19.2MB/s]
User cancelled operation
^C


In [ ]:
DATASET_DIR = RAW_DATA_DIR / "plantvillage dataset" / "color"

## 3. Dataset Quality Audit (Targeting 'color' subset)

Auditing the raw dataset with a focus strictly on the **RGB color subset** (~54,303 images across 38 classes):
- **Subset Isolation**: Automatically locates and isolates the `color` directory to avoid scanning redundant `grayscale` and `segmented` variants.
- **Data Integrity Check**: Validates file headers and decodability with PIL to flag corrupt/unreadable images.
- **Class Imbalance & Geometry Profiling**: Gathers resolution, aspect ratio, color mode, and image distribution across all 38 plant disease classes.
- **Export Reports**: Saves `corrupt_images.txt` and `class_distribution.csv` inside `AUDIT_DIR`.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# 1. Locate the 'color' directory
color_candidates = list(RAW_DATA_DIR.rglob("color"))
if not color_candidates:
    raise FileNotFoundError(f"Could not find a 'color' directory inside {RAW_DATA_DIR}")

# Select the target directory containing the class subfolders
COLOR_DATA_DIR = next(p for p in color_candidates if p.is_dir())
print(f"Targeting color subset directory: {COLOR_DATA_DIR}")

# 2. Gather all image file paths
valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
color_images = [
    p for p in COLOR_DATA_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in valid_extensions
]

print(f"Total image files discovered in 'color': {len(color_images):,}")

# 3. Perform integrity checks and extract metadata
corrupt_files = []
records = []

for img_path in tqdm(color_images, desc="Auditing 'color' images"):
    # Immediate parent directory serves as the class label
    class_name = img_path.parent.name
    
    try:
        # Verify structural integrity of the image file
        with Image.open(img_path) as img:
            img.verify()
        
        # Reopen to read dimensions and color mode (verify() resets the stream)
        with Image.open(img_path) as img:
            width, height = img.size
            mode = img.mode
            
        file_size_kb = round(img_path.stat().st_size / 1024, 2)
        
        records.append({
            "path": str(img_path.relative_to(PROJECT_ROOT)),
            "class_name": class_name,
            "width": width,
            "height": height,
            "mode": mode,
            "size_kb": file_size_kb
        })
        
    except Exception as e:
        corrupt_files.append((str(img_path), str(e)))

# 4. Save corrupt files log
corrupt_report_path = AUDIT_DIR / "corrupt_images.txt"
with open(corrupt_report_path, "w", encoding="utf-8") as f:
    for path, err in corrupt_files:
        f.write(f"{path}\t{err}\n")

# 5. Build metadata DataFrame and export aggregated report
df_meta = pd.DataFrame(records)

if not df_meta.empty:
    class_summary = (
        df_meta.groupby("class_name")
        .agg(
            image_count=("path", "count"),
            min_width=("width", "min"),
            max_width=("width", "max"),
            min_height=("height", "min"),
            max_height=("height", "max"),
            avg_size_kb=("size_kb", "mean")
        )
        .reset_index()
        .sort_values(by="image_count", ascending=False)
    )
    
    csv_report_path = AUDIT_DIR / "class_distribution.csv"
    class_summary.to_csv(csv_report_path, index=False)
    
    print("\n" + "=" * 50)
    print("            AUDIT SUMMARY ('color')            ")
    print("=" * 50)
    print(f"Total valid images : {len(df_meta):,}")
    print(f"Total corrupt files: {len(corrupt_files)}")
    print(f"Total classes      : {df_meta['class_name'].nunique()}")
    print(f"Color modes found  : {df_meta['mode'].unique().tolist()}")
    print("\nTop 5 classes with most samples:")
    print(class_summary.head(5)[["class_name", "image_count"]].to_string(index=False))
    print("\nBottom 5 classes with fewest samples (potential imbalance):")
    print(class_summary.tail(5)[["class_name", "image_count"]].to_string(index=False))
    print(f"\nAudit artifacts exported to: {AUDIT_DIR}")
else:
    print("Warning: No valid images found in the target directory.")

/home/tanthanh/Documents/Projects/PlantDiseaseClassification/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Targeting color subset directory: /home/tanthanh/Documents/Projects/PlantDiseaseClassification/data/raw/plantvillage dataset/color
Total image files discovered in 'color': 54,305


Auditing 'color' images: 100%|██████████| 54305/54305 [00:09<00:00, 5675.03it/s]



            AUDIT SUMMARY ('color')            
Total valid images : 54,305
Total corrupt files: 0
Total classes      : 38
Color modes found  : ['RGB', 'RGBA']

Top 5 classes with most samples:
                              class_name  image_count
Orange___Haunglongbing_(Citrus_greening)         5507
  Tomato___Tomato_Yellow_Leaf_Curl_Virus         5357
                       Soybean___healthy         5090
                  Peach___Bacterial_spot         2297
                 Tomato___Bacterial_spot         2127

Bottom 5 classes with fewest samples (potential imbalance):
                  class_name  image_count
Tomato___Tomato_mosaic_virus          373
         Raspberry___healthy          371
             Peach___healthy          360
    Apple___Cedar_apple_rust          275
            Potato___healthy          152

Audit artifacts exported to: /home/tanthanh/Documents/Projects/PlantDiseaseClassification/data/audit
